In [19]:
# ============================================
# SMART CLASSROOM AUTOMATION SYSTEM
# 01 - DATA GENERATION
# ============================================

import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

print("Data generation notebook started.")

Data generation notebook started.


In [20]:
# ============================================
# LOAD REAL ENERGY DATA
# ============================================

DATABASE_PATH = Path("../data/classroom.db")

connection = sqlite3.connect(
    DATABASE_PATH
)

real_data = pd.read_sql_query(
    """
    SELECT *
    FROM energy_logs
    """,
    connection
)

connection.close()

print("Real dataset shape:", real_data.shape)

real_data.head()

Real dataset shape: (24, 8)


,id,timestamp,zone_name,appliance,people_count,duration_seconds,power_watts,energy_wh
0,1,2026-09-29T02:38:11.185845,Zone 1,light,2,60.000000,40.0,0.666667
1,2,2026-09-29T02:38:11.188136,Zone 1,fan,2,60.000000,75.0,1.250000
2,3,2026-09-29T02:43:57.272710,Zone 2,light,0,17.856521,40.0,0.198406
3,4,2026-09-29T02:43:57.284567,Zone 2,fan,0,15.834768,75.0,0.329891
4,5,2026-09-29T02:44:13.127614,Zone 1,light,0,16.857750,40.0,0.187308


In [21]:
# ============================================
# MARK REAL DATA
# ============================================

real_data["data_source"] = "real"

print(
    real_data["data_source"].value_counts()
)

data_source
real    24
Name: count, dtype: int64


In [22]:
# ============================================
# SIMULATION SETTINGS
# ============================================

np.random.seed(42)

NUMBER_OF_SESSIONS = 2000

ZONES = [
    "Zone 1",
    "Zone 2"
]

APPLIANCES = [
    "light",
    "fan"
]

LIGHT_POWER_WATTS = 40
FAN_POWER_WATTS = 75

print(
    "Simulated sessions:",
    NUMBER_OF_SESSIONS
)

Simulated sessions: 2000


In [23]:
# ============================================
# GENERATE OCCUPANCY
# ============================================

people_counts = []

for _ in range(NUMBER_OF_SESSIONS):

    # Empty classroom is common
    if np.random.random() < 0.20:

        people = 0

    else:

        # Most occupied sessions have moderate occupancy
        people = int(
            np.clip(
                np.random.normal(
                    loc=18,
                    scale=10
                ),
                1,
                40
            )
        )

    people_counts.append(people)


people_counts = np.array(
    people_counts
)

print("Minimum people:", people_counts.min())
print("Maximum people:", people_counts.max())
print("Average people:", people_counts.mean())

Minimum people: 0
Maximum people: 40
Average people: 14.088


In [24]:
# ============================================
# GENERATE ZONE AND APPLIANCE
# ============================================

simulated_data = pd.DataFrame({

    "zone_name": np.random.choice(
        ZONES,
        NUMBER_OF_SESSIONS
    ),

    "appliance": np.random.choice(
        APPLIANCES,
        NUMBER_OF_SESSIONS
    ),

    "people_count": people_counts
})

simulated_data.head()

,zone_name,appliance,people_count
0,Zone 1,fan,6
1,Zone 1,fan,0
2,Zone 2,fan,0
3,Zone 1,fan,21
4,Zone 2,fan,12


In [25]:
# ============================================
# GENERATE SESSION DURATIONS
# ============================================

base_duration = np.random.gamma(
    shape=2.5,
    scale=20,
    size=NUMBER_OF_SESSIONS
)

occupancy_effect = (
    simulated_data["people_count"].values
    * np.random.uniform(
        0.5,
        2.0,
        NUMBER_OF_SESSIONS
    )
)

duration_seconds = (
    base_duration
    + occupancy_effect
)

duration_seconds = np.clip(
    duration_seconds,
    3,
    300
)

simulated_data["duration_seconds"] = (
    duration_seconds
)

simulated_data["duration_seconds"] = (
    simulated_data["duration_seconds"]
    .round(2)
)

simulated_data.head()

,zone_name,appliance,people_count,duration_seconds
0,Zone 1,fan,6,31.24
1,Zone 1,fan,0,6.81
2,Zone 2,fan,0,140.01
3,Zone 1,fan,21,57.74
4,Zone 2,fan,12,147.24


In [26]:
# ============================================
# ASSIGN POWER RATINGS
# ============================================

simulated_data["power_watts"] = np.where(
    simulated_data["appliance"] == "light",
    LIGHT_POWER_WATTS,
    FAN_POWER_WATTS
)

simulated_data.head()

,zone_name,appliance,people_count,duration_seconds,power_watts
0,Zone 1,fan,6,31.24,75
1,Zone 1,fan,0,6.81,75
2,Zone 2,fan,0,140.01,75
3,Zone 1,fan,21,57.74,75
4,Zone 2,fan,12,147.24,75


In [27]:
# ============================================
# CALCULATE ENERGY
# ============================================

simulated_data["energy_wh"] = (
    simulated_data["power_watts"]
    * simulated_data["duration_seconds"]
    / 3600
)

simulated_data["energy_wh"] = (
    simulated_data["energy_wh"]
    .round(6)
)

simulated_data.head()

,zone_name,appliance,people_count,duration_seconds,power_watts,energy_wh
0,Zone 1,fan,6,31.24,75,0.650833
1,Zone 1,fan,0,6.81,75,0.141875
2,Zone 2,fan,0,140.01,75,2.916875
3,Zone 1,fan,21,57.74,75,1.202917
4,Zone 2,fan,12,147.24,75,3.067500


In [28]:
# ============================================
# GENERATE TIMESTAMPS
# ============================================

start_date = pd.Timestamp(
    "2026-01-01 06:00:00"
)

random_seconds = np.random.randint(
    0,
    90 * 24 * 60 * 60,
    NUMBER_OF_SESSIONS
)

simulated_data["timestamp"] = (
    start_date
    + pd.to_timedelta(
        random_seconds,
        unit="s"
    )
)

simulated_data["timestamp"] = (
    simulated_data["timestamp"]
    .sort_values()
    .values
)

simulated_data.head()

,zone_name,appliance,people_count,duration_seconds,power_watts,energy_wh,timestamp
0,Zone 1,fan,6,31.24,75,0.650833,2026-01-01 06:27:38
1,Zone 1,fan,0,6.81,75,0.141875,2026-01-01 08:09:09
2,Zone 2,fan,0,140.01,75,2.916875,2026-01-01 08:15:09
3,Zone 1,fan,21,57.74,75,1.202917,2026-01-01 08:30:12
4,Zone 2,fan,12,147.24,75,3.067500,2026-01-01 08:34:57


In [29]:
# ============================================
# MARK SIMULATED DATA
# ============================================

simulated_data["data_source"] = "simulated"

print(
    simulated_data["data_source"].value_counts()
)

data_source
simulated    2000
Name: count, dtype: int64


In [30]:
# ============================================
# COMBINE DATASETS
# ============================================

combined_data = pd.concat(
    [
        real_data,
        simulated_data
    ],
    ignore_index=True
)

print(
    "Real rows:",
    len(real_data)
)

print(
    "Simulated rows:",
    len(simulated_data)
)

print(
    "Combined rows:",
    len(combined_data)
)

Real rows: 24
Simulated rows: 2000
Combined rows: 2024


In [31]:
# ============================================
# DATASET SUMMARY
# ============================================

print(
    combined_data.info()
)

print()
print(
    combined_data["data_source"]
    .value_counts()
)

print()
print(
    combined_data.describe()
)

<class 'pandas.DataFrame'>
RangeIndex: 2024 entries, 0 to 2023
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                24 non-null     float64
 1   timestamp         2024 non-null   object 
 2   zone_name         2024 non-null   str    
 3   appliance         2024 non-null   str    
 4   people_count      2024 non-null   int64  
 5   duration_seconds  2024 non-null   float64
 6   power_watts       2024 non-null   float64
 7   energy_wh         2024 non-null   float64
 8   data_source       2024 non-null   str    
dtypes: float64(4), int64(1), object(1), str(3)
memory usage: 142.4+ KB
None

data_source
simulated    2000
real           24
Name: count, dtype: int64

              id  people_count  duration_seconds  power_watts    energy_wh
count  24.000000   2024.000000        2024.00000  2024.000000  2024.000000
mean   12.500000     13.927866          66.74579    57.742095     1.074634
std     7.

In [32]:
# ============================================
# OCCUPANCY CHECK
# ============================================

print(
    "Minimum occupancy:",
    combined_data["people_count"].min()
)

print(
    "Maximum occupancy:",
    combined_data["people_count"].max()
)

print(
    "Average occupancy:",
    combined_data["people_count"].mean()
)

Minimum occupancy: 0
Maximum occupancy: 40
Average occupancy: 13.92786561264822


In [33]:
# ============================================
# VERIFY ENERGY CALCULATION
# ============================================

calculated_energy = (
    combined_data["power_watts"]
    * combined_data["duration_seconds"]
    / 3600
)

difference = (
    combined_data["energy_wh"]
    - calculated_energy
).abs()

print(
    "Maximum calculation difference:",
    difference.max()
)

Maximum calculation difference: 4.4444444458058285e-07


In [34]:
# ============================================
# SAVE SIMULATED DATA
# ============================================

PROCESSED_DIR = Path(
    "../data/processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

simulated_path = (
    PROCESSED_DIR /
    "simulated_energy_data.csv"
)

simulated_data.to_csv(
    simulated_path,
    index=False
)

print(
    "Simulated dataset saved:"
)

print(simulated_path)

Simulated dataset saved:
..\data\processed\simulated_energy_data.csv


In [35]:
# ============================================
# SAVE COMBINED DATASET
# ============================================

combined_path = (
    PROCESSED_DIR /
    "combined_energy_data.csv"
)

combined_data.to_csv(
    combined_path,
    index=False
)

print(
    "Combined dataset saved:"
)

print(combined_path)

Combined dataset saved:
..\data\processed\combined_energy_data.csv


In [36]:
# ============================================
# FINAL SUMMARY
# ============================================

print()
print("============================================")
print("01 DATA GENERATION COMPLETE")
print("============================================")

print()
print("Real sessions     :", len(real_data))
print("Simulated sessions:", len(simulated_data))
print("Combined sessions :", len(combined_data))

print()
print("Occupancy range:")
print(
    combined_data["people_count"].min(),
    "to",
    combined_data["people_count"].max(),
    "people"
)

print()
print("Files created:")
print("data/processed/simulated_energy_data.csv")
print("data/processed/combined_energy_data.csv")


01 DATA GENERATION COMPLETE

Real sessions     : 24
Simulated sessions: 2000
Combined sessions : 2024

Occupancy range:
0 to 40 people

Files created:
data/processed/simulated_energy_data.csv
data/processed/combined_energy_data.csv
